In [1]:

import nltk
from nltk.corpus import movie_reviews
import random


nltk.download('movie_reviews')

[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\37498\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\movie_reviews.zip.


True

In [2]:

documents = [(list(movie_reviews.words(fileid)), category)
             for category in movie_reviews.categories()
             for fileid in movie_reviews.fileids(category)]


random.shuffle(documents)


print(documents[:2])


[(['in', 'roger', 'michell', "'", 's', 'romantic', 'comedy', 'notting', 'hill', ',', 'william', 'thacker', '(', 'hugh', 'grant', ')', 'leads', 'a', 'rather', 'dreary', 'life', 'maintaining', 'his', 'flagging', 'travel', 'bookshop', 'in', 'the', 'quaint', 'section', 'of', 'london', 'which', 'lends', 'it', "'", 's', 'name', 'to', 'the', 'film', "'", 's', 'title', '.', 'one', 'day', ',', 'american', 'movie', 'superstar', 'anna', 'scott', '(', 'julia', 'roberts', ')', 'walks', 'in', 'to', 'purchase', 'a', 'book', 'on', 'turkey', '.', 'quickly', 'enamored', 'of', 'each', 'other', ',', 'the', 'two', 'embark', 'upon', 'an', 'on', '-', 'again', ',', 'off', '-', 'again', 'love', 'affair', 'replete', 'with', 'romance', ',', 'humor', ',', 'and', 'the', 'occasional', 'lump', 'in', 'the', 'throat', '.', 'the', 'film', 'opens', 'with', 'a', 'non', '-', 'verbal', 'cue', 'to', 'anna', "'", 's', 'stardom', 'as', 'the', 'title', 'credits', 'appear', 'over', 'a', 'montage', 'of', 'slow', 'motion', 'seque

In [3]:

X_text = [" ".join(words) for words, label in documents]
y = [label for words, label in documents]
print(X_text[:2])
print(y[:2])

['in roger michell \' s romantic comedy notting hill , william thacker ( hugh grant ) leads a rather dreary life maintaining his flagging travel bookshop in the quaint section of london which lends it \' s name to the film \' s title . one day , american movie superstar anna scott ( julia roberts ) walks in to purchase a book on turkey . quickly enamored of each other , the two embark upon an on - again , off - again love affair replete with romance , humor , and the occasional lump in the throat . the film opens with a non - verbal cue to anna \' s stardom as the title credits appear over a montage of slow motion sequences featuring the actress \' s appearances in films and at premieres - coming out of limousines , walking the red carpets and such . without words , this sequence gives us a background to her character . following , however , is a set - up narration by william indicating what he does and where he lives . i don \' t know why the filmmakers chose to go with a narration wh

In [4]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

Training samples: 1600
Testing samples: 400


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('clf', LogisticRegression(max_iter=1000))
])

pipeline.fit(X_train, y_train)
print("Model training completed!")


Model training completed!


In [6]:
from sklearn.metrics import accuracy_score
y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.2f}")

Test Accuracy: 0.84


In [7]:

misclassified_count = 0

for i, (true_label, pred_label) in enumerate(zip(y_test, y_pred)):
    if true_label != pred_label:
        print(f"\nReview #{i}:")
        print("Text:", X_test[i][:500], "...")  
        print("True label:", true_label)
        print("Predicted label:", pred_label)
        
        misclassified_count += 1
        if misclassified_count >= 2:
            break


Review #4:
Text: when i left the theater after seeing david lynch ' s " lost highway , " i remarked to a fellow movie - goer , " i feel like someone just sucked my brains out through my nose and put them back in through my ears . " in his first feature film in five years , lynch delivers a film second only to his debut picture " eraserhead " on the weirdness scale . you won ' t " know " what happened when it ' s over , though there are certainly enough clues with which to make some reasonable guesses . it is dif ...
True label: pos
Predicted label: neg

Review #7:
Text: let ' s face it : since waterworld floated by , the summer movie season has grown * very * stale . with no new eye - candy for four weeks straight , we ' ve had to sustain ourselves on the quasi - nutritional value of cheatin ' husbands , traveling chocolate salesmen , and computer - generated serial killers . sigh . thank god for desperado . the freewheeling sequel to el mariachi -- director robert rodriguez ' s notor

# Module 6: Sentiment Analysis with Movie Reviews

*Prepared by: Alisa Serobyan*

---

## Exercise 1: Conceptual Questions
### 1 Model Failure
A lexicon-based model incorrectly classifies the sentence:
*"This concert was anything but boring"*
- Why it fails: Lexicon-based models only look at individual words. The word "boring" is negative, so the model marks the whole sentence as negative.  
- How ML can fix it: A machine learning model trained on similar sentences would learn patterns like "anything but X" flips the sentiment. It uses context and word combinations, not just single words.

### 2 Data Requirements
Task: Build a sentiment analyzer for a highly technical software product.
- Best approach: Machine learning-based.  
- Reason: Lexicon-based methods would fail because of domain-specific jargon not in general sentiment dictionaries.  
- Main challenge: Need enough labeled data with the technical terms to train the model effectively. Without sufficient examples, the model might not understand specialized vocabulary.

### 3 Beyond Positive/Negative
Task: Multi-class sentiment analysis: "Happy", "Sad", "Angry", "Surprised"
- Pipeline changes:
  1. Use a multi-class classifier, e.g., LogisticRegression(multi_class='multinomial', solver='lbfgs')  
  2. Vectorization (e.g., `TfidfVectorizer`) stays the same.  
  3. Evaluation metrics: Use accuracy per class, precision/recall, or confusion matrix to see how well each emotion is detected.  
- Key idea: The model now predicts one of several classes instead of just positive/negative.

## Exercise 2: Movie Review Classifier
- Steps followed:
  1. Loaded data from nltk.corpus.movie_reviews and shuffled documents.
  2. Converted each review from list of words → string (`X_text`) and created labels list (`y`).
  3. Train/test split: 80% training, 20% testing.
  4. Created a Pipeline with TfidfVectorizer(stop_words='english') + LogisticRegression(max_iter=1000).
  5. Trained the model with .fit(X_train, y_train).
  6. Evaluated the model on the test set using .predict(X_test) and accuracy_score.
- Result: Model achieved good accuracy (~0.8–0.85 typical).

## Exercise 3: Challenge Problem – Error Analysis
- Found at least two reviews misclassified by the model.
- Printed review text, true label, and predicted label.
- Analysis: Common reasons for misclassification:
  1. Negation: e.g., "not good" may confuse the model.
  2. Sarcasm: e.g., "Great, another boring sequel."
  3. Unusual vocabulary: film-specific names or rare words not seen in training.
- This demonstrates the real-world challenge of NLP: understanding why the model fails is as important as building it.